# String Output Parser

In [9]:
from langchain_mistralai import ChatMistralAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
model= ChatMistralAI(model="mistral-medium-2508")

In [6]:
template1= PromptTemplate(
    template=" Write a detailed report on {topic}",
    input_variables= ["topic"]
)

template2= PromptTemplate(
    template=" Write a 2 sentence summary on the following text. \n {text}",
    input_variables= ["text"]
)

In [ ]:
# this is the process without output parser and chaining

prompt1= template1.invoke({'topic': 'Theory of Relativity'})
result1= model.invoke(prompt1)

prompt2= template2.invoke({'text': result1.content}) # without parser, we need to manually pass the result with .content to extract text
result2= model.invoke(prompt2)

print(result2.content)

Albert Einstein’s **Theory of Relativity**, comprising **Special Relativity (1905)** and **General Relativity (1915)**, revolutionized physics by redefining space, time, and gravity—showing that time is relative, gravity curves spacetime, and mass and energy are interchangeable (*E=mc²*). Experimentally confirmed through phenomena like **time dilation (GPS), gravitational waves (LIGO), and black holes (Event Horizon Telescope)**, it remains foundational to modern physics while posing unresolved challenges like **quantum gravity and dark energy**.


In [10]:
parser= StrOutputParser()

chain= template1 | model | template2 | model | parser

result= chain.invoke({'topic': 'Theory of Relativity'})
print(result)

Albert Einstein’s **Theory of Relativity**, comprising **Special Relativity (1905)** and **General Relativity (1915)**, revolutionized physics by redefining space, time, and gravity—introducing concepts like time dilation, spacetime curvature, and mass-energy equivalence (\\(E=mc^2\\)). The theory has been experimentally confirmed through phenomena like gravitational waves, black holes, and GPS technology, though challenges like quantum gravity and dark matter remain unresolved.


# Json Output Parser

In [11]:
from langchain_mistralai import ChatMistralAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from dotenv import load_dotenv

load_dotenv()

True

In [13]:
model= ChatMistralAI(model="mistral-medium-2508")

parser= JsonOutputParser()

In [25]:
template= PromptTemplate(
    template= "Give me name, age and city of a fictional person and explain in one sentence why you chose it. \n{format_instructions}",
    input_variables=[],
    partial_variables= {'format_instructions': parser.get_format_instructions}
)

chain= template | model | parser
result= chain.invoke({})
print(result, "\n", type(result))

{'fictional_person': {'name': 'Elara Voss', 'age': 28, 'city': 'Port Velen', 'reason': 'I chose *Elara Voss* for its sleek, futuristic sound, *28* as a prime age for ambition and mystery, and *Port Velen*—a fictional coastal city—to evoke a blend of maritime intrigue and cyberpunk energy, perfect for a character caught between tradition and innovation.'}} 
 <class 'dict'>


# Structured Output Parser

In [34]:
from langchain_mistralai import ChatMistralAI
from langchain_core.prompts import PromptTemplate
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from dotenv import load_dotenv

load_dotenv()

True

In [35]:
model= ChatMistralAI(model="mistral-medium-2508")

In [40]:
schema= [
    ResponseSchema(name='fact 1', description= 'Fact 1 about the topic'),
    ResponseSchema(name='fact 2', description= 'Fact 2 about the topic'),
    ResponseSchema(name='fact 3', description= 'Fact 3 about the topic')
]

parser= StructuredOutputParser.from_response_schemas(schema)

template= PromptTemplate(
    template= 'Give 3 facts about the {topic} \n{format_instructions}',
    input_variables= ['topic'],
    partial_variables= {'format_instructions': parser.get_format_instructions}
)

prompt= template.invoke({'topic': 'Wormhole'})
result= model.invoke(prompt)
print(result.content)

```json
{
	"fact 1": "Wormholes are hypothetical structures connecting two separate points in spacetime, often described as 'tunnels' that could allow for near-instantaneous travel between distant locations in the universe, as predicted by Einstein's theory of general relativity.",

	"fact 2": "The concept of wormholes was first theorized in 1916 by physicist Ludwig Flamm, who explored solutions to Einstein's equations that allowed for such 'bridges' (later called Einstein-Rosen bridges in 1935 when Einstein and Nathan Rosen expanded on the idea).",

	"fact 3": "For a wormhole to remain stable and traversable, it would likely require 'exotic matter' with negative energy to counteract the gravitational forces that would otherwise collapse it—a substance that has never been observed in nature and remains purely theoretical."
}
```


# Pydantic Output Parser

In [46]:
from langchain_mistralai import ChatMistralAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from dotenv import load_dotenv

load_dotenv()

True

In [45]:
model= ChatMistralAI(model="mistral-medium-2508")

In [51]:
class Person(BaseModel):
    
    name: str= Field(description="Name of the person.")
    age: int= Field(gt=18, description="Age of the person.")
    city: str= Field(description="Name of the city the person belongs to")
    
parser= PydanticOutputParser(pydantic_object= Person)

template= PromptTemplate(
    template='Generate name, age and city of a fictional {place} person \n{format_instructions}',
    input_variables= ['place'],
    partial_variables= {'format_instructions': parser.get_format_instructions}
) 

chain= template | model | parser

result= chain.invoke({'place': 'tropical'})

print(result)

name='Kaiani Moku' age=29 city='Hanapepe, Kauaʻi'
